# Bronze → Silver: ocorrencias_rede

**Entrada:** `` `v-energy`.bronze.tb_ocorrencias_rede `` (texto bruto, 180.000 linhas).
**Saída:** `` `v-energy`.silver.tb_ocorrencias_rede `` (Delta, tipada e padronizada).
**Grão:** uma linha por ocorrência (`id_ocorrencia` único). Pela definição do case, cada registro é **uma UC interrompida**.

**O que este notebook faz.** Cada passo corresponde a um achado de `docs/auditoria_dados/auditoria_ocorrencias_rede.md`:
1. **Tipa as datas** como `TIMESTAMP_NTZ` (hora local de Recife, sem fuso) e `duracao_min` como inteiro, removendo o sufixo `" min"`. A **`competencia`** é o mês de abertura, como `DATE` no primeiro dia do mês (o eixo de tempo do glossário e do contrato da Gold).
2. **Padroniza `causa`:** as 4 formas de ausência (`''`, `-`, `N/A`, `NULL`) viram a categoria explícita `NAO INFORMADA`, separada de `NAO IDENTIFICADA`.
3. **Trata as interrupções sobrepostas na mesma UC, dentro de cada competência:** `flag_sobreposicao` e `duracao_efetiva_min`, que conta só os minutos ainda não cobertos por uma interrupção anterior da UC na mesma competência. Somar `duracao_efetiva_min` de qualquer conjunto de competências, com ou sem o evento atípico, dá o tempo real sem energia, sem contar nada em dobro.
4. **Marca o evento atípico por alimentador e competência** (`flag_evento_atipico`): competência com mais de `LIMIAR_EVENTO_ATIPICO` vezes a base mensal do alimentador. Hoje marca exatamente o evento climático de 2025-07.
5. **Valida antes de gravar:** se uma regra crítica falhar, o notebook para e a Silver atual não é tocada. A validação e a escrita leem a mesma versão da Bronze. Carga completa com `overwrite` (idempotente). Depois de gravar, confere a tabela e, se ela não bater com o que foi validado, volta a Silver para a versão anterior.

Documentação: `docs/silver/silver_ocorrencias_rede.md`.

In [ ]:
# ruff: noqa: F821
# (spark e display são variáveis globais do notebook no Databricks)
catalog = 'v-energy'
bronze_schema = 'bronze'
silver_schema = 'silver'

bronze_table = f'`{catalog}`.{bronze_schema}.tb_ocorrencias_rede'
silver_table = f'`{catalog}`.{silver_schema}.tb_ocorrencias_rede'

COLUNAS_BRONZE = [
    'id_ocorrencia',
    'alimentador',
    'id_uc',
    'tipo',
    'causa',
    'data_abertura',
    'data_encerramento',
    'duracao_min',
]
COLUNAS_LINHAGEM = ['ingestion_datetime', 'source_file']
TIPOS_VALIDOS = [
    'FALTA DE ENERGIA',
    'DANO EM EQUIPAMENTO',
    'OSCILACAO',
    'SOBRECARGA',
    'FIO PARTIDO',
]
CAUSA_NAO_INFORMADA = 'NAO INFORMADA'
CAUSAS_VALIDAS = [
    'CLIMATICA',
    'SOBRECARGA',
    'NAO IDENTIFICADA',
    'VEGETACAO',
    'TERCEIROS',
    'FALHA DE EQUIPAMENTO',
    CAUSA_NAO_INFORMADA,
]
MARCADORES_AUSENCIA = ['', '-', 'N/A', 'NULL']  # formas de "sem causa" na auditoria (seção 3.2)
PADRAO_DATA_HORA = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'  # formato único da origem
FUSO_ORIGEM = 'America/Recife'  # os horários da origem são locais; "agora" é calculado neste fuso
# Evento atípico: mês com mais de LIMIAR_EVENTO_ATIPICO vezes a base mensal do alimentador.
# Com menos de MESES_MINIMOS_SERIE meses, a base do alimentador é a mediana da rede.
LIMIAR_EVENTO_ATIPICO = 5
MESES_MINIMOS_SERIE = 12

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.{silver_schema}')

## 1. Leitura da Bronze
Fixa a versão atual da Bronze (`VERSION AS OF`): a validação e a escrita leem exatamente os mesmos dados, mesmo que a Bronze seja recarregada durante a execução. Depois confere se as colunas esperadas existem.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

versao_bronze = spark.sql(f'DESCRIBE HISTORY {bronze_table} LIMIT 1').first()['version']
df_bronze = spark.sql(f'SELECT * FROM {bronze_table} VERSION AS OF {versao_bronze}')
faltando = set(COLUNAS_BRONZE) - set(df_bronze.columns)
assert not faltando, f'Colunas ausentes na Bronze: {faltando}'

linhas_bronze = df_bronze.count()
print(f'{bronze_table} (versão {versao_bronze}): {linhas_bronze} linhas')

## 2. Padronização e tipagem
- **Datas:** o texto no formato da origem (`yyyy-MM-dd HH:mm:ss`) vira `TIMESTAMP_NTZ` por `try_cast`, que não passa pelo fuso da sessão: o horário gravado é sempre o da origem, mesmo numa sessão com horário de verão. Outro formato ou data inexistente vira nulo e é barrado na validação. As contas de minutos usam `minutos_entre()`, a diferença entre os dois horários locais, porque o `timestampdiff` converte para o fuso da sessão.
- **`duracao_min`:** extrai os dígitos (aceita `"217"` e `"217 min"`) e converte com `try_cast`.
- **`causa`:** maiúsculas, sem acento, e ausência → `NAO INFORMADA`.
- **`sigla_municipio`** (`REC`, `JAB`…) vem do código do alimentador, e **`competencia`** (primeiro dia do mês de abertura) da data de abertura.
- As duplicatas exatas (as mesmas colunas da Bronze) são removidas depois da padronização. Fica a linha da carga mais antiga (`ingestion_datetime` e, no empate, `source_file`), então a linhagem gravada não muda entre execuções. A Bronze atual não tem nenhuma, mas a regra protege as próximas cargas.

In [ ]:
def texto(coluna):
    """Remove espaços nas pontas, converte para maiúsculas e tira acentos."""
    return F.translate(
        F.upper(F.trim(F.col(coluna))), 'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ', 'AAAAAEEEEIIIIOOOOOUUUUC'
    )


def para_data_hora(coluna):
    """Texto no formato da origem → TIMESTAMP_NTZ, sem passar pelo fuso da sessão.

    O cast de texto para TIMESTAMP_NTZ não usa fuso, mas aceita outros formatos (só a data, com
    T...), então o padrão restringe ao da origem. Data inexistente vira nulo.
    """
    convertida = F.expr(f'try_cast(trim({coluna}) AS TIMESTAMP_NTZ)')
    return F.when(F.trim(F.col(coluna)).rlike(PADRAO_DATA_HORA), convertida)


def minutos_entre(inicio, fim):
    """Minutos de inicio a fim (TIMESTAMP_NTZ), pela diferença entre os horários locais.

    O timestampdiff converte os dois para TIMESTAMP no fuso da sessão: num fuso com horário de
    verão, um horário que cai no salto do relógio muda a conta em 60 minutos.
    """
    return F.expr(f'CAST(CAST({fim} - {inicio} AS INTERVAL MINUTE) AS BIGINT)')


causa = texto('causa')

df_padronizado = (
    df_bronze.select(
        texto('id_ocorrencia').alias('id_ocorrencia'),
        texto('id_uc').alias('id_uc'),
        texto('alimentador').alias('alimentador'),
        texto('tipo').alias('tipo'),
        F.when(causa.isNull() | causa.isin(MARCADORES_AUSENCIA), F.lit(CAUSA_NAO_INFORMADA))
        .otherwise(causa)
        .alias('causa'),
        para_data_hora('data_abertura').alias('data_abertura'),
        para_data_hora('data_encerramento').alias('data_encerramento'),
        # "217", "217 min" ou "217 MIN" → "217"; outro formato vira nulo e é barrado na validação
        F.regexp_extract(F.col('duracao_min'), r'(?i)^\s*(\d+)\s*(min)?\s*$', 1).alias(
            '_duracao_texto'
        ),
        *COLUNAS_LINHAGEM,
    )
    .withColumn('duracao_min', F.expr("try_cast(nullif(_duracao_texto, '') AS INT)"))
    .withColumn('sigla_municipio', F.regexp_extract('alimentador', r'^ALM-([A-Z]{3})-\d{3}$', 1))
    # DATE sem fuso: a data local da abertura, no primeiro dia do mês
    .withColumn('competencia', F.trunc(F.col('data_abertura').cast('date'), 'month'))
    .drop('_duracao_texto')
)

# Duplicata exata: as mesmas colunas da Bronze, depois da padronização. Fica a linha da carga mais
# antiga, então a linhagem gravada (source_file, ingestion_datetime) é sempre a mesma
mais_antiga = Window.partitionBy(*COLUNAS_BRONZE).orderBy(*COLUNAS_LINHAGEM)
df_unico = (
    df_padronizado.withColumn('_ordem', F.row_number().over(mais_antiga))
    .where('_ordem = 1')
    .drop('_ordem')
)

## 3. Interrupções sobrepostas na mesma UC
Para cada ocorrência, `maior_fim_anterior` é o **maior encerramento** entre as ocorrências anteriores da mesma UC **abertas na mesma competência**. Comparar só com a imediatamente anterior não basta, porque uma interrupção longa pode conter uma curta e uma terceira.
- `flag_sobreposicao`: a ocorrência começa antes de `maior_fim_anterior`, ou seja, a UC já estava sem energia.
- `duracao_efetiva_min`: minutos desta ocorrência que **ainda não** estavam cobertos (`encerramento − maior_fim_anterior`, ou zero). Para ocorrências sem sobreposição, é igual a `duracao_min`.

Assim, `SUM(duracao_efetiva_min)` é o tempo real sem energia (união dos intervalos, base do DEC), e `COUNT_IF(NOT flag_sobreposicao)` é o número de interrupções distintas (base do FEC).

**Por que dentro da competência:** a Gold soma por competência e escolhe se inclui o evento atípico. Se uma ocorrência de outra competência, ou de uma competência de evento atípico, cobrisse esta, os minutos ficariam com a outra e sumiriam quando a Gold tirasse aquela competência da conta. A janela é por UC e competência, e uma regra crítica garante que a UC está num único alimentador na competência. Como o flag de evento atípico vale para o alimentador na competência inteira, a janela nunca mistura ocorrências com e sem o flag. O custo é pequeno: uma interrupção que atravessa a virada do mês e cobre outra do mês seguinte conta nos dois meses, em minutos e como uma interrupção a mais no FEC do mês seguinte (na extração atual, 1 caso e 34 min).

**Por causa:** os minutos em comum ficam com a interrupção que já estava em curso, então a soma por causa reparte o tempo sem energia e fecha com o total. Filtrar uma causa não recalcula o tempo das outras sem ela (ver a seção 2.4 do doc).

In [ ]:
# Janela por UC e competência. A regra crítica de um alimentador por UC na competência mantém a
# janela dentro de um alimentador, onde o flag de evento atípico vale para a competência inteira:
# a Gold pode somar qualquer conjunto de competências, com ou sem o evento, sem perder minutos
por_uc_competencia = Window.partitionBy('id_uc', 'competencia')
anteriores_na_competencia = por_uc_competencia.orderBy(
    'data_abertura', 'data_encerramento', 'id_ocorrencia'
).rowsBetween(Window.unboundedPreceding, -1)
minutos_nao_cobertos = minutos_entre('_maior_fim_anterior', 'data_encerramento')

df_sobreposicao = (
    df_unico.withColumn(
        '_maior_fim_anterior', F.max('data_encerramento').over(anteriores_na_competencia)
    )
    .withColumn(
        'flag_sobreposicao',
        F.coalesce(F.col('data_abertura') < F.col('_maior_fim_anterior'), F.lit(False)),
    )
    .withColumn(
        'duracao_efetiva_min',
        F.when(F.col('flag_sobreposicao'), F.greatest(F.lit(0), minutos_nao_cobertos))
        .otherwise(F.col('duracao_min'))
        .cast('int'),
    )
    # Para a validação: a UC aparece em mais de um alimentador na competência (mesma partição)
    .withColumn(
        '_varios_alimentadores',
        F.min('alimentador').over(por_uc_competencia)
        != F.max('alimentador').over(por_uc_competencia),
    )
    .drop('_maior_fim_anterior')
)

## 4. Evento atípico por alimentador e competência
Para cada alimentador, conta as ocorrências de cada mês do calendário **entre a primeira e a última competência em que ele aparece**. A grade é gerada com `sequence`, então um mês sem nenhuma ocorrência entra com zero. Uma competência é de **evento atípico** quando passa de `LIMIAR_EVENTO_ATIPICO` vezes a **base mensal** do alimentador:
- **base** = mediana mensal do próprio alimentador, mas nunca abaixo da **mediana da rede** (todas as combinações alimentador × competência). O piso impede que um alimentador quase sem ocorrências tenha marcado qualquer mês com 6 ou mais.
- Com menos de `MESES_MINIMOS_SERIE` meses, a base é a mediana da rede. Numa série curta, a mediana do próprio alimentador inclui o evento: com os meses [30, 6.887], ela daria 3.458 e o evento passaria.

O intervalo de cada alimentador importa para os que entram ou saem no meio do período: se os meses em que ele ainda não existia (ou já tinha saído) entrassem com zero, a mediana dele iria a zero.

Na extração atual, os 122 alimentadores aparecem nos 48 meses, a mediana da rede é 28 e as medianas dos alimentadores vão de 25 a 29,5. Os 3 alimentadores do evento climático de 2025-07 ficam entre **93× e 246×** a base, e todos os outros meses ficam em no máximo **2,3×**. O limite de 5× separa os dois grupos com folga, sem data fixa no código. O flag não remove nada: a Gold decide se calcula DEC/FEC com ou sem essas competências.

In [ ]:
por_alimentador_mes = df_sobreposicao.groupBy('alimentador', 'competencia').agg(
    F.count('*').alias('ocorrencias')
)

# Calendário completo entre a primeira e a última competência de cada alimentador
grade = (
    por_alimentador_mes.groupBy('alimentador')
    .agg(
        F.sequence(F.min('competencia'), F.max('competencia'), F.expr('INTERVAL 1 MONTH')).alias(
            '_meses'
        )
    )
    .select('alimentador', F.explode('_meses').alias('competencia'))
)
serie_mensal = grade.join(por_alimentador_mes, ['alimentador', 'competencia'], 'left').fillna(
    0, ['ocorrencias']
)

# Base do alimentador: a própria mediana, nunca abaixo da mediana da rede (nem de 1)
mediana_rede = serie_mensal.agg(F.median('ocorrencias')).first()[0]
piso = F.lit(max(mediana_rede or 0, 1))
base = (
    serie_mensal.groupBy('alimentador')
    .agg(F.median('ocorrencias').alias('mediana_mensal'), F.count('*').alias('meses'))
    .withColumn(
        'base_mensal',
        F.when(F.col('meses') < MESES_MINIMOS_SERIE, piso).otherwise(
            F.greatest('mediana_mensal', piso)
        ),
    )
)

meses_avaliados = serie_mensal.join(base, 'alimentador').withColumn(
    'flag_evento_atipico', F.col('ocorrencias') > LIMIAR_EVENTO_ATIPICO * F.col('base_mensal')
)
meses_atipicos = meses_avaliados.where('flag_evento_atipico').select(
    'alimentador', 'competencia', 'flag_evento_atipico'
)

df_tratado = (
    df_sobreposicao.join(F.broadcast(meses_atipicos), ['alimentador', 'competencia'], 'left')
    .fillna(False, ['flag_evento_atipico'])
    .withColumn('silver_datetime', F.current_timestamp())
)

## 5. Validação antes de gravar
Cada regra conta as linhas que a **violam**, e todas são calculadas numa única passada. O id repetido entra na mesma passada como `count − count distinct` (linhas a mais), sem janela.
- **crítica:** interrompe o notebook antes do `overwrite`, e a Silver atual não é tocada. Antes de parar, o notebook mostra até 20 linhas que violam cada regra.
- **aviso:** é exibida, mas não bloqueia a carga.

"Agora" é a hora atual em `FUSO_ORIGEM`, o mesmo fuso das datas da origem. A sessão do Databricks costuma estar em UTC, 3 horas à frente.

In [ ]:
# Agora no fuso da origem, como TIMESTAMP_NTZ (a sessão costuma estar em UTC, 3 horas à frente)
agora = F.expr(f"convert_timezone('{FUSO_ORIGEM}', current_timestamp())")

duracao_pelas_datas = minutos_entre('data_abertura', 'data_encerramento')

REGRA_ID_REPETIDO = 'id_ocorrencia repetido após remover duplicatas exatas (linhas a mais)'
REGRAS = [  # (regra, condição que identifica uma linha que VIOLA a regra, crítica?)
    (
        'id_ocorrencia ou id_uc fora do padrão OC…/UC… + dígitos',
        F.col('id_ocorrencia').isNull()
        | ~F.col('id_ocorrencia').rlike(r'^OC\d+$')
        | F.col('id_uc').isNull()
        | ~F.col('id_uc').rlike(r'^UC\d+$'),
        True,
    ),
    (
        'alimentador fora do padrão ALM-AAA-999',
        F.col('sigla_municipio').isNull() | (F.col('sigla_municipio') == ''),
        True,
    ),
    ('id_uc em mais de um alimentador na mesma competência', F.col('_varios_alimentadores'), True),
    ('tipo fora do domínio', F.col('tipo').isNull() | ~F.col('tipo').isin(TIPOS_VALIDOS), True),
    ('causa fora do domínio', ~F.col('causa').isin(CAUSAS_VALIDAS), True),
    ('data_abertura vazia ou não convertida', F.col('data_abertura').isNull(), True),
    ('data_encerramento vazia ou não convertida', F.col('data_encerramento').isNull(), True),
    (
        'encerramento anterior à abertura',
        F.col('data_encerramento') < F.col('data_abertura'),
        True,
    ),
    (
        'abertura ou encerramento no futuro',
        F.greatest('data_abertura', 'data_encerramento') > agora,
        True,
    ),
    ('duracao_min vazia ou não convertida', F.col('duracao_min').isNull(), True),
    (
        'duracao_min diferente de encerramento − abertura',
        F.col('duracao_min') != duracao_pelas_datas,
        True,
    ),
    (
        'duracao_efetiva_min fora do intervalo [0, duracao_min]',
        F.col('duracao_efetiva_min').isNull()
        | (F.col('duracao_efetiva_min') < 0)
        | (F.col('duracao_efetiva_min') > F.col('duracao_min')),
        True,
    ),
]

contagens = df_tratado.agg(
    F.count('*').alias('linhas'),
    (F.count('id_ocorrencia') - F.count_distinct('id_ocorrencia')).alias('ids_repetidos'),
    F.sum('duracao_min').alias('minutos_somados'),
    F.sum('duracao_efetiva_min').alias('minutos_efetivos'),
    F.count_if('flag_sobreposicao').alias('sobrepostas'),
    F.count_if('flag_evento_atipico').alias('em_evento_atipico'),
    *[F.count_if(condicao).alias(f'regra_{i}') for i, (_, condicao, _) in enumerate(REGRAS)],
).first()

linhas_silver = contagens['linhas']
somados, efetivos = contagens['minutos_somados'], contagens['minutos_efetivos']
resultado = [(REGRA_ID_REPETIDO, int(contagens['ids_repetidos']), 'crítica')] + [
    (regra, int(contagens[f'regra_{i}']), 'crítica' if critica else 'aviso')
    for i, (regra, _, critica) in enumerate(REGRAS)
]
display(
    spark.createDataFrame(
        [
            (r, v, n, 'OK' if v == 0 else ('FALHA' if n == 'crítica' else 'ATENÇÃO'))
            for r, v, n in resultado
        ],
        'regra string, violacoes long, nivel string, status string',
    )
)
print(
    f'Bronze: {linhas_bronze} linhas | Silver: {linhas_silver} linhas | '
    f'duplicatas removidas: {linhas_bronze - linhas_silver}'
)
print(
    f'Sobrepostas: {contagens["sobrepostas"]} | minutos somados: {somados} | '
    f'minutos efetivos: {efetivos} | em dobro: {somados - efetivos}'
)
print(f'Ocorrências em evento atípico: {contagens["em_evento_atipico"]}')
print(f'Mediana da rede (alimentador × competência): {mediana_rede}')
display(
    meses_avaliados.where('flag_evento_atipico')
    .select('alimentador', 'competencia', 'ocorrencias', 'mediana_mensal', 'base_mensal')
    .orderBy('competencia', 'alimentador')
)

# Exemplos das regras críticas que falharam (só são calculados nesse caso)
ids_repetidos = df_tratado.groupBy('id_ocorrencia').count().where('count > 1').drop('count')
exemplos = {REGRA_ID_REPETIDO: df_tratado.join(ids_repetidos, 'id_ocorrencia')}
exemplos.update({regra: df_tratado.where(condicao) for regra, condicao, _ in REGRAS})
falhas = [f'{r}: {v} linha(s)' for r, v, n in resultado if v and n == 'crítica']
for regra, violacoes, nivel in resultado:
    if nivel == 'crítica' and violacoes:
        print(f'Exemplos de linhas que violam: {regra}')
        display(exemplos[regra].limit(20))
assert not falhas, 'Silver de ocorrências reprovada na validação:\n- ' + '\n- '.join(falhas)

## 6. Escrita na Silver
Os comentários de coluna vão no schema (`withMetadata`) e são gravados no mesmo commit da carga. O do `flag_evento_atipico` é montado a partir das constantes da regra, então acompanha qualquer mudança nelas.

Antes de gravar, o notebook guarda a versão atual da Silver. Depois, confere se a tabela gravada tem as colunas esperadas e o número de linhas validado. Se não tiver, a Silver volta para essa versão (`RESTORE`), ou é removida se esta foi a primeira carga, e o notebook para. O comentário da tabela só é reescrito depois dessa conferência e quando muda; o texto vai escapado (`literal_sql`).

In [ ]:
COMENTARIOS = {
    'id_ocorrencia': (
        'Identificador da ocorrência (chave da tabela). Cada ocorrência é uma UC interrompida.'
    ),
    'id_uc': 'Unidade consumidora interrompida.',
    'alimentador': 'Alimentador de média tensão (ALM-<sigla do município>-<nnn>).',
    'sigla_municipio': 'Sigla do município extraída do alimentador (REC, JAB, OLI...).',
    'tipo': (
        'Tipo da ocorrência (FALTA DE ENERGIA, FIO PARTIDO, OSCILACAO, SOBRECARGA, '
        'DANO EM EQUIPAMENTO).'
    ),
    'causa': (
        'Causa registrada; NAO INFORMADA quando a origem não trouxe causa '
        '(diferente de NAO IDENTIFICADA).'
    ),
    'data_abertura': 'Início da interrupção, hora local de Recife (TIMESTAMP_NTZ, sem fuso).',
    'data_encerramento': 'Fim da interrupção, hora local de Recife (TIMESTAMP_NTZ, sem fuso).',
    'competencia': 'Competência da abertura: o primeiro dia do mês de data_abertura.',
    'duracao_min': 'Duração registrada em minutos (= encerramento - abertura).',
    'duracao_efetiva_min': (
        'Minutos desta ocorrência não cobertos por ocorrência anterior da mesma UC na mesma '
        'competência. Use SUM desta coluna para o tempo sem energia (DEC). Por causa, os minutos '
        'em comum ficam com a interrupção que já estava em curso.'
    ),
    'flag_sobreposicao': (
        'True se a ocorrência começou enquanto a UC já estava sem energia por outra ocorrência '
        'aberta na mesma competência. Para contar interrupções (FEC), use as linhas com False.'
    ),
    'flag_evento_atipico': (
        'True se a ocorrência foi aberta numa competência de evento atípico do alimentador: mais '
        f'de {LIMIAR_EVENTO_ATIPICO}x a base mensal dele (a mediana mensal do alimentador, nunca '
        f'abaixo da mediana da rede; com menos de {MESES_MINIMOS_SERIE} meses, a mediana da rede).'
    ),
    'source_file': 'Arquivo de origem na landing.',
    'ingestion_datetime': 'Momento da carga na Bronze.',
    'silver_datetime': 'Momento da carga na Silver.',
}
COLUNAS_SILVER = list(COMENTARIOS)
COMENTARIO_TABELA = (
    'Silver: ocorrências de rede tipadas e padronizadas, uma linha por UC interrompida, com '
    'tratamento de sobreposição e marcação de evento atípico. '
    'Regras em docs/silver/silver_ocorrencias_rede.md.'
)


def literal_sql(texto):
    """Texto entre aspas simples para SQL, com barra invertida e apóstrofo escapados."""
    return "'" + texto.replace('\\', '\\\\').replace("'", "\\'") + "'"


df_final = df_tratado.select(*COLUNAS_SILVER)
for coluna, comentario in COMENTARIOS.items():
    df_final = df_final.withMetadata(coluna, {'comment': comentario})

# Versão atual da Silver, para voltar a ela se a conferência depois da escrita falhar
versao_anterior = (
    spark.sql(f'DESCRIBE HISTORY {silver_table} LIMIT 1').first()['version']
    if spark.catalog.tableExists(silver_table)
    else None
)

(
    df_final.write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(silver_table)
)

In [ ]:
# Depois de gravar: a tabela tem as colunas esperadas e o volume validado acima. Se não tiver, a
# Silver volta para a versão anterior (ou é removida, na primeira carga) e o notebook para
tabela = spark.table(silver_table)
colunas_tabela, linhas_tabela = tabela.columns, tabela.count()

if colunas_tabela != COLUNAS_SILVER or linhas_tabela != linhas_silver:
    if versao_anterior is None:
        spark.sql(f'DROP TABLE {silver_table}')
    else:
        spark.sql(f'RESTORE TABLE {silver_table} TO VERSION AS OF {versao_anterior}')
    raise AssertionError(
        f'Silver gravada com {linhas_tabela} linhas e colunas {colunas_tabela}; esperado '
        f'{linhas_silver} linhas e {COLUNAS_SILVER}. A carga foi desfeita '
        f'(versão anterior: {versao_anterior}).'
    )

if spark.sql(f'DESCRIBE DETAIL {silver_table}').first()['description'] != COMENTARIO_TABELA:
    spark.sql(f'COMMENT ON TABLE {silver_table} IS {literal_sql(COMENTARIO_TABELA)}')
print(f'OK: {linhas_tabela} ocorrências carregadas em {silver_table}')

## Evidências: antes × depois e uso na Gold
As consultas abaixo comparam a Bronze lida no passo 1 (mesma versão) com a Silver e mostram como a Gold deve usar as colunas novas. O `id_ocorrencia` da Bronze passa pela mesma padronização da Silver antes do join. Elas não alteram dados.

In [ ]:
# Bronze lida no passo 1 (mesma versão), com o id padronizado como na Silver
df_bronze.withColumn('id_silver', texto('id_ocorrencia')).createOrReplaceTempView('bronze_lida')
b, s = 'bronze_lida', silver_table

# 1. causa: valores da Bronze → Silver
display(
    spark.sql(f"""
  SELECT coalesce(b.causa, '<null>') AS causa_bronze, s.causa AS causa_silver,
         COUNT(*) AS ocorrencias
  FROM {b} b JOIN {s} s ON b.id_silver = s.id_ocorrencia
  GROUP BY 1, 2 ORDER BY 3 DESC""")
)

# 2. duracao_min: com sufixo "min" (em qualquer caixa) ou só dígitos na Bronze → inteiro na Silver
display(
    spark.sql(f"""
  SELECT CASE WHEN b.duracao_min RLIKE '(?i)min' THEN 'com sufixo min' ELSE 'só dígitos' END
           AS formato_bronze,
         COUNT(*) AS ocorrencias, SUM(s.duracao_min) AS minutos_silver
  FROM {b} b JOIN {s} s ON b.id_silver = s.id_ocorrencia
  GROUP BY 1""")
)

# 3. Tempo sem energia: soma direta × efetiva (sem contar sobreposição em dobro)
display(
    spark.sql(f"""
  SELECT COUNT(*) AS ocorrencias,
         COUNT_IF(NOT flag_sobreposicao) AS interrupcoes_distintas,
         SUM(duracao_min) AS minutos_somados,
         SUM(duracao_efetiva_min) AS minutos_efetivos,
         SUM(duracao_min) - SUM(duracao_efetiva_min) AS minutos_em_dobro
  FROM {s}""")
)

# 4. Prévia da Gold: numeradores de DEC/FEC por alimentador, com e sem o evento atípico
#    (o denominador, nº de UCs do alimentador, vem do cadastro)
display(
    spark.sql(f"""
  SELECT alimentador,
         ROUND(SUM(duracao_efetiva_min) / 60, 1) AS horas_sem_energia,
         COUNT_IF(NOT flag_sobreposicao) AS interrupcoes,
         ROUND(SUM(CASE WHEN NOT flag_evento_atipico THEN duracao_efetiva_min END) / 60, 1)
           AS horas_sem_evento_atipico,
         COUNT_IF(NOT flag_sobreposicao AND NOT flag_evento_atipico)
           AS interrupcoes_sem_evento_atipico
  FROM {s} GROUP BY 1 ORDER BY 2 DESC LIMIT 10""")
)